### News

Introducing **Unsloth Studio** — a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Load model + tokenizer

In [2]:
from unsloth import FastLanguageModel, FastModel
import torch

MODEL = "unsloth/gemma-3-270m-it"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = False,
    full_finetuning = False
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.15: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

### Define LoRA Config

In [3]:
model = FastModel.get_peft_model(
    model,
    r = 128,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    use_gradient_checkpointing = "unsloth",
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Making `model.base_model.model.model` require gradients


<a name="Data"></a>
### Data Prep
We'll use a sampled dataset of handwritten math formulas. The objective is to convert these images into a computer-readable format—specifically LaTeX—so they can be rendered. This is particularly useful for complex expressions.

You can access the dataset [here](https://huggingface.co/datasets/unsloth/LaTeX_OCR). The full dataset is [here](https://huggingface.co/datasets/linxy/LaTeX_OCR).

In [4]:
from datasets import load_dataset

dataset_name = "Thytu/ChessInstruct"
dataset = load_dataset(dataset_name, split = "train[:10000]")

README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/161M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.63M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/99000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Let's take an overview of the dataset. We'll examine the second image and its corresponding caption.

In [5]:
dataset

Dataset({
    features: ['task', 'input', 'expected_output', 'KIND'],
    num_rows: 10000
})

In [6]:
dataset[0]

{'task': "Given an incomplit set of chess moves and the game's final score, write the last missing chess move.\n\nInput Format: A comma-separated list of chess moves followed by the game score.\nOutput Format: The missing chess move",
 'input': '{"moves": ["d2d4", "g8f6", "c2c4", "e7e6", "b1c3", "f8b4", "d1c2", "d7d5", "a2a3", "b4c3", "c2c3", "f6e4", "c3c2", "e8g8", "g1f3", "b7b6", "c1f4", "c7c5", "e2e3", "c8b7", "d4c5", "b6c5", "e1c1", "b8c6", "f1d3", "d8b6", "d3e4", "d5e4", "f3e5", "c6e5", "f4e5", "f7f6", "e5c3", "a8d8", "d1d8", "f8d8", "c2a4", "b7c6", "a4a5", "g8f7", "c1c2", "e6e5", "b2b3", "d8d7", "a5b6", "a7b6", "a3a4", "d7a7", "h1a1", "f7e7", "a4a5", "b6a5", "c3a5", "e7e6", "c2b2", "h7h5", "a1a3", "a7a6", "a5c7", "c6b7", "a3a5", "a6a5", "c7a5", "e6d6", "b3b4", "c5b4", "a5b4", "d6d7", "b2c3", "b7c8", "b4f8", "g7g6", "c3b4", "h5h4", "b4b5", "c8b7", "f8g7", "f6f5", "g7e5", "b7c6", "b5c5", "h4h3", "g2h3", "c6a8", "e5d6", "a8c6", "d6f8", "c6b7", "c5d4", "d7e6", "h3h4", "b7c6", "f8h6",

### Prepare Dataset

In [7]:
from unsloth import to_sharegpt
from unsloth.chat_templates import standardize_data_formats

dataset = load_dataset("Thytu/ChessInstruct", split = "train")

dataset = standardize_data_formats(dataset)

def convert_to_chatml(example):
    return {
        "conversations": [
            {"role": "system", "content": example["task"]},
            {"role": "user", "content": example["input"]},
            {"role": "assistant", "content": example["expected_output"]}
        ]
    }

dataset = dataset.map(
    convert_to_chatml
)

Map:   0%|          | 0/99000 [00:00<?, ? examples/s]

In [8]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="gemma3-chess-lora",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=100,
        learning_rate=5e-5,
        logging_steps=1,
        save_steps=50,
        save_total_limit=2,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        warmup_steps=5,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
        seed=3407,
        max_length=max_seq_length,
        packing=False,
    ),
)

NameError: name 'max_seq_length' is not defined

### Trainer

In [ ]:
trainer_stats = trainer.train()

In [ ]:
model.save_pretrained("gemma3-chess-lora")
tokenizer.save_pretrained("gemma3-chess-lora")

In [ ]:
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name="gemma3-chess-lora",  # your saved path
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
)

model.eval()

In [9]:
messages = [
    {"role": "system", "content": "You are a chess expert."},
    {"role": "user", "content": "What is the best opening for white?"}
]

In [10]:
inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True,   # important for inference
).to(model.device)

In [11]:
outputs = model.generate(
    inputs,
    max_new_tokens=200,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [12]:
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

user
You are a chess expert.

What is the best opening for white?
model
That's a great question! There's no single "best" opening for white, as it depends on the specific situation and the player's style. However, I can give you some strong contenders that often work well in various chess positions:

*   **1. e4:** This is a classic and often a very strong opening. It's a solid foundation for many positions, and it can be played with a variety of pieces.
*   **2. d4:** This is a very common and versatile opening. It's often used in both kingside and queenside positions.
*   **3. Nf3:** This is a very popular and often a good choice for white. It's a solid opening that can be played with a variety of pieces.
*   **4. d4 d5:** This is a very common and versatile opening. It's often used in both kingside and queenside positions.
*   
